# 01 — Universe Audit

**Goal:** Understand what's in `universe.csv` before migrating to the `stocks` table.

**Questions to answer:**
1. How many stocks per cap_tier?
2. What columns exist? Which map to the `stocks` schema?
3. Any duplicate sids? Missing required fields?
4. Sector distribution — any gaps?
5. How do nifty500_list.csv and slug_map.csv relate?

In [ ]:
import pandas as pd
from pathlib import Path

V1 = Path.home() / "alpha-signal"

# Load the main universe file
uni = pd.read_csv(V1 / "data/harvester/universe.csv")
print(f"Shape: {uni.shape}")
print(f"\nColumns ({len(uni.columns)}):")
for c in uni.columns:
    print(f"  {c:25s}  {uni[c].dtype}  nulls={uni[c].isnull().sum()}")


In [ ]:
from itables import show
show(uni.head())

In [ ]:
# Tier distribution + duplicate check
print("=== Cap Tier Distribution ===")
print(uni['cap_tier'].value_counts().sort_index())
print(f"\n=== Duplicate SIDs: {uni['sid'].duplicated().sum()} ===")
print(f"=== Duplicate Tickers: {uni['ticker'].duplicated().sum()} ===")

# Sector distribution
print(f"\n=== Sectors ({uni['sector'].nunique()}) ===")
sec = uni['sector'].value_counts()
print(sec.to_string())
print(f"\nSectors with < 5 stocks: {(sec < 5).sum()}")


In [ ]:
# Inspect the 52 ETFs — these shouldn't be in a stock universe
print("=== ETFs (sample) ===")
print(uni[uni['sector'] == 'ETF'][['sid', 'ticker', 'name', 'cap_tier']].head(15).to_string(index=False))

# Where do the market_cap nulls land?
print(f"\n=== market_cap nulls by tier ===")
print(uni[uni['market_cap'].isnull()]['cap_tier'].value_counts())

print(f"\n=== adtv_6m_cr nulls by tier ===")
print(uni[uni['adtv_6m_cr'].isnull()]['cap_tier'].value_counts())


In [ ]:
# Match nifty500 via ticker column instead of sid
nifty_tickers = set(nifty['Symbol'].str.strip())
uni_in_nifty = uni['ticker'].isin(nifty_tickers)
print(f"Nifty500 matched via ticker: {uni_in_nifty.sum()} of {len(nifty_tickers)}")

# What about the existing in_nifty500 column — is it already correct?
print(f"\nExisting in_nifty500=True count: {uni['in_nifty500'].sum()}")
print(f"Match between existing flag and ticker match: {(uni['in_nifty500'] == uni_in_nifty).sum()}")

# Unmatched nifty500 tickers (not in universe)
unmatched = nifty_tickers - set(uni['ticker'])
print(f"\nNifty500 tickers NOT in universe: {len(unmatched)}")
if unmatched:
    print(sorted(unmatched)[:20])


In [ ]:
# Check slug_map
slugs = pd.read_csv(V1 / "data/harvester/slug_map.csv")
print(f"=== slug_map.csv: {slugs.shape} ===")
print(f"Columns: {list(slugs.columns)}")
print(slugs.head(3).to_string(index=False))

# How many universe stocks have slugs?
slug_match = uni['sid'].isin(slugs['sid'] if 'sid' in slugs.columns else slugs.iloc[:,0])
print(f"\nUniverse stocks with slugs: {slug_match.sum()} of {len(uni)}")
print(f"Slugs not in universe: {len(slugs) - slug_match.sum()}")


In [ ]:
# === COLUMN MAPPING: universe.csv → stocks table ===
# This is what we'll use during migration

print("=== universe.csv columns → stocks table ===")
print("""
  universe.csv          →  stocks table          Notes
  ─────────────────────────────────────────────────────
  sid                   →  sid (PK)              Direct
  name                  →  name                  Direct
  ticker                →  ticker                Direct
  sector                →  sector                Direct (after filtering ETFs)
  in_nifty500           →  in_nifty500           Rebuild from nifty500_list.csv
  market_cap            →  market_cap_cr         Rename (778 nulls, all SMALL)
  adtv_6m_cr            →  adtv_6m_cr            Direct (507 nulls, mostly SMALL)
  market_cap_rank       →  (not in schema)       Used to derive cap_tier, then drop
  cap_tier              →  cap_tier              Direct

  slug_map.csv:
  slug                  →  slug                  Merge on sid

  NOT in universe.csv (leave NULL for now, populate later):
  industry, pe_ratio, pb_ratio, roe, debt_to_equity,
  dividend_yield, revenue_growth, profit_margin,
  free_cashflow, beta, fifty_two_week_high,
  fifty_two_week_low, avg_volume
""")

# Quick sanity: after ETF filter, what's left?
clean = uni[uni['sector'] != 'ETF']
print(f"After removing ETFs: {len(clean)} stocks")
print(f"Tier distribution:\n{clean['cap_tier'].value_counts().sort_index()}")


In [ ]:
import sys
sys.path.insert(0, str(Path.home() / "alpha-signal-v2"))
from db import get_db, upsert_df, read_table

# Filter ETFs
clean = uni[uni['sector'] != 'ETF'].copy()

# Merge slugs
slugs_df = slugs[['sid', 'slug']].drop_duplicates('sid')
clean = clean.merge(slugs_df, on='sid', how='left')

# Rebuild in_nifty500 from fresh nifty500 list
nifty_tickers = set(nifty['Symbol'].str.strip())
clean['in_nifty500'] = clean['ticker'].isin(nifty_tickers).astype(int)

# Rename to match schema
clean = clean.rename(columns={'market_cap': 'market_cap_cr'})

# Select only columns that exist in stocks table
stocks_cols = ['sid', 'ticker', 'name', 'sector', 'cap_tier',
               'market_cap_cr', 'adtv_6m_cr', 'in_nifty500', 'slug']
stocks_df = clean[stocks_cols]

print(f"Ready to insert: {len(stocks_df)} rows")
print(f"Columns: {list(stocks_df.columns)}")
print(f"\nSample:")
print(stocks_df.head(5).to_string(index=False))


In [ ]:
# Write to SQLite
rows = upsert_df(stocks_df, "stocks")
print(f"Inserted: {rows} rows")

# Verify
result = read_table("stocks")
print(f"\nVerification — stocks table: {len(result)} rows")
print(f"Tier distribution:\n{result['cap_tier'].value_counts().sort_index()}")
print(f"\nin_nifty500 count: {result['in_nifty500'].sum()}")
print(f"Slugs populated: {result['slug'].notna().sum()}")
print(f"Sectors: {result['sector'].nunique()}")


# Session 1.2: Fundamental Data Audit

In [ ]:
import pandas as pd
from pathlib import Path

V1 = Path.home() / "alpha-signal"

# Load all 4 fundamental CSVs
qi = pd.read_csv(V1 / "data/harvester/quarterly_income.csv")
bs = pd.read_csv(V1 / "data/harvester/annual_balancesheet.csv")
cf = pd.read_csv(V1 / "data/harvester/annual_cashflow.csv")
sh = pd.read_csv(V1 / "data/harvester/shareholding.csv")

for name, df in [("quarterly_income", qi), ("annual_balancesheet", bs), 
                  ("annual_cashflow", cf), ("shareholding", sh)]:
    print(f"=== {name}: {df.shape} ===")
    print(f"  Columns: {list(df.columns)}")
    print()


In [ ]:
# === COLUMN MAPPING ANALYSIS ===

print("=== quarterly_income: CSV → schema ===")
print("""
  CSV column          → schema column       Action
  ──────────────────────────────────────────────────
  sid                 → sid                 Direct
  period              → period              Direct
  end_date            → end_date            Direct
  reporting           → reporting           Direct
  revenue             → revenue             Direct
  net_income          → net_income          Direct
  eps                 → eps                 Direct
  operating_profit    → operating_profit    Direct
  raw_material_cost   → (NOT IN SCHEMA)     DROP or ADD?
  sga_expense         → (NOT IN SCHEMA)     DROP or ADD?
  interest_expense    → interest            RENAME
  pbt                 → (NOT IN SCHEMA)     DROP or ADD?
  total_other_income  → (NOT IN SCHEMA)     DROP or ADD?
  dps                 → (NOT IN SCHEMA)     DROP or ADD?
  name                → (DROP)              Already in stocks table
  
  SCHEMA HAS BUT CSV LACKS:
  ebitda, tax, depreciation
""")

# Check what pbt, raw_material_cost etc look like — are they useful?
print("=== Extra columns null rates ===")
for col in ['raw_material_cost', 'sga_expense', 'pbt', 'total_other_income', 'dps']:
    null_pct = qi[col].isnull().sum() / len(qi) * 100
    print(f"  {col:25s}  {null_pct:5.1f}% null")


In [ ]:
print("=== annual_balancesheet: CSV → schema ===")
print("""
  CSV column                  → schema column           Action
  ───────────────────────────────────────────────────────────────
  sid                         → sid                     Direct
  period                      → period                  Direct
  end_date                    → end_date                Direct
  total_assets                → total_assets             Direct
  total_equity                → total_equity             Direct
  total_debt                  → total_debt               Direct
  total_current_assets        → current_assets           RENAME
  total_current_liabilities   → current_liabilities      RENAME
  cash                        → cash_and_equivalents     RENAME
  total_receivables           → receivables              RENAME
  retained_earnings           → retained_earnings        Direct
  name                        → (DROP)
  
  CSV HAS, SCHEMA LACKS:
  net_ppe, total_liabilities, shares_outstanding, long_term_debt
  
  SCHEMA HAS, CSV LACKS:
  inventory, goodwill
""")

# Check usefulness of extra BS columns
print("=== Extra BS columns null rates ===")
for col in ['net_ppe', 'total_liabilities', 'shares_outstanding', 'long_term_debt']:
    null_pct = bs[col].isnull().sum() / len(bs) * 100
    print(f"  {col:25s}  {null_pct:5.1f}% null")

print("\n=== annual_cashflow: CSV → schema ===")
print("""
  CSV column                  → schema column           Action
  ───────────────────────────────────────────────────────────────
  sid                         → sid                     Direct
  period                      → period                  Direct
  end_date                    → end_date                Direct
  operating_cash_flow         → operating_cash_flow     Direct
  capex                       → capex                   Direct
  free_cash_flow              → free_cash_flow          Direct
  investing_cash_flow         → investing_cash_flow     Direct
  financing_cash_flow         → financing_cash_flow     Direct
  name                        → (DROP)
  
  CSV HAS, SCHEMA LACKS:
  working_capital_change, depreciation, interest_expense_cf, net_change_in_cash
  
  SCHEMA HAS, CSV LACKS:
  dividends_paid
""")

# Check usefulness of extra CF columns
print("=== Extra CF columns null rates ===")
for col in ['working_capital_change', 'depreciation', 'interest_expense_cf', 'net_change_in_cash']:
    null_pct = cf[col].isnull().sum() / len(cf) * 100
    print(f"  {col:25s}  {null_pct:5.1f}% null")


In [ ]:
print("=== shareholding: CSV → schema ===")
print("""
  CSV column            → schema column       Action
  ─────────────────────────────────────────────────────
  sid                   → sid                 Direct
  end_date              → end_date            Direct
  promoter_pct          → promoter_pct        Direct
  fii_pct               → fii_pct             Direct
  mf_pct                → mf_pct              Direct
  dii_pct               → dii_pct             Direct
  public_pct            → public_pct          Direct
  promoter_pledged_pct  → pledge_pct          RENAME
  name                  → (DROP)

  CSV HAS, SCHEMA LACKS:
  insurance_pct, retail_hni_pct, other_pct
""")

# Check extra shareholding columns
for col in ['insurance_pct', 'retail_hni_pct', 'other_pct']:
    null_pct = sh[col].isnull().sum() / len(sh) * 100
    print(f"  {col:25s}  {null_pct:5.1f}% null")

# === DATA QUALITY: orphan SIDs (in CSV but not in stocks table) ===
import sys
sys.path.insert(0, str(Path.home() / "alpha-signal-v2"))
from db import read_table
stocks = read_table("stocks")
valid_sids = set(stocks['sid'])

print("\n=== Orphan SIDs (in CSV but NOT in stocks table) ===")
for name, df in [("quarterly_income", qi), ("annual_balancesheet", bs), 
                  ("annual_cashflow", cf), ("shareholding", sh)]:
    orphans = set(df['sid']) - valid_sids
    print(f"  {name:25s}  {len(orphans)} orphans")


In [ ]:
# === PERIOD FORMATS — what do they look like? ===
print("=== quarterly_income periods (sample) ===")
print(sorted(qi['period'].unique())[:15])
print(f"Total unique periods: {qi['period'].nunique()}")

print("\n=== annual_balancesheet periods (sample) ===")
print(sorted(bs['period'].unique())[:15])
print(f"Total unique periods: {bs['period'].nunique()}")

print("\n=== shareholding end_dates (sample) ===")
print(sorted(sh['end_date'].unique())[:10])
print(f"Total unique dates: {sh['end_date'].nunique()}")

# === KEY NULL RATES for critical fields ===
print("\n=== Critical field null rates ===")
checks = [
    ("qi.revenue",           qi['revenue']),
    ("qi.net_income",        qi['net_income']),
    ("qi.eps",               qi['eps']),
    ("bs.total_assets",      bs['total_assets']),
    ("bs.total_equity",      bs['total_equity']),
    ("bs.total_debt",        bs['total_debt']),
    ("cf.operating_cash_flow", cf['operating_cash_flow']),
    ("cf.free_cash_flow",    cf['free_cash_flow']),
    ("sh.promoter_pct",      sh['promoter_pct']),
]
for label, series in checks:
    null_pct = series.isnull().sum() / len(series) * 100
    print(f"  {label:30s}  {null_pct:5.1f}% null")


In [ ]:
# Shareholding: how many rows have the 1899 date?
bad_dates = sh[sh['end_date'] == '1899-12-31']
print(f"=== Shareholding rows with 1899-12-31: {len(bad_dates)} ===")
if len(bad_dates) > 0:
    print(bad_dates[['sid', 'end_date', 'promoter_pct']].head(5).to_string(index=False))

# How many quarters per stock?
print(f"\n=== Shareholding: quarters per stock ===")
quarters_per_stock = sh.groupby('sid').size()
print(quarters_per_stock.describe())

# How many stocks per quarter (top 10 most recent)?
print(f"\n=== Stocks per shareholding date (most recent 10) ===")
print(sh.groupby('end_date').size().sort_index().tail(10))

# Quarterly income: quarters per stock
print(f"\n=== Quarterly income: periods per stock ===")
qi_per_stock = qi.groupby('sid').size()
print(qi_per_stock.describe())

# Annual BS: years per stock
print(f"\n=== Annual balance sheet: years per stock ===")
bs_per_stock = bs.groupby('sid').size()
print(bs_per_stock.describe())


In [ ]:
# Re-insert stocks (DB was reinitialized with updated schema)
upsert_df(stocks_df, "stocks")
print(f"stocks: {len(read_table('stocks'))} rows restored")

# === MIGRATE QUARTERLY INCOME ===
qi_clean = qi[qi['sid'].isin(valid_sids)].copy()
qi_clean = qi_clean.rename(columns={'interest_expense': 'interest'})
qi_cols = ['sid', 'period', 'end_date', 'reporting', 'revenue', 'operating_profit',
           'net_income', 'eps', 'interest', 'pbt', 'total_other_income']
qi_insert = qi_clean[qi_cols]
qi_rows = upsert_df(qi_insert, "quarterly_income")
print(f"quarterly_income: {qi_rows} rows inserted")

# === MIGRATE ANNUAL BALANCE SHEET ===
bs_clean = bs[bs['sid'].isin(valid_sids)].copy()
bs_clean = bs_clean.rename(columns={
    'total_current_assets': 'current_assets',
    'total_current_liabilities': 'current_liabilities',
    'cash': 'cash_and_equivalents',
    'total_receivables': 'receivables'
})
bs_cols = ['sid', 'period', 'end_date', 'total_assets', 'total_equity', 'total_debt',
           'current_assets', 'current_liabilities', 'cash_and_equivalents', 'receivables',
           'retained_earnings', 'net_ppe', 'total_liabilities', 'shares_outstanding', 'long_term_debt']
bs_insert = bs_clean[bs_cols]
bs_rows = upsert_df(bs_insert, "annual_balance_sheet")
print(f"annual_balance_sheet: {bs_rows} rows inserted")

# === MIGRATE ANNUAL CASH FLOW ===
cf_clean = cf[cf['sid'].isin(valid_sids)].copy()
cf_cols = ['sid', 'period', 'end_date', 'operating_cash_flow', 'capex', 'free_cash_flow',
           'investing_cash_flow', 'financing_cash_flow', 'working_capital_change',
           'depreciation', 'net_change_in_cash']
cf_insert = cf_clean[cf_cols]
cf_rows = upsert_df(cf_insert, "annual_cash_flow")
print(f"annual_cash_flow: {cf_rows} rows inserted")

# === MIGRATE SHAREHOLDING ===
sh_clean = sh[(sh['sid'].isin(valid_sids)) & (sh['end_date'] != '1899-12-31')].copy()
sh_clean = sh_clean.rename(columns={'promoter_pledged_pct': 'pledge_pct'})
sh_cols = ['sid', 'end_date', 'promoter_pct', 'fii_pct', 'mf_pct', 'dii_pct',
           'public_pct', 'pledge_pct', 'insurance_pct', 'retail_hni_pct', 'other_pct']
sh_insert = sh_clean[sh_cols]
sh_rows = upsert_df(sh_insert, "shareholding")
print(f"shareholding: {sh_rows} rows inserted (7 bad-date rows filtered)")


In [ ]:
# What values are violating the constraint?
print("=== other_pct outliers ===")
bad = sh_clean[~sh_clean['other_pct'].between(0, 100)]
print(f"Rows with other_pct outside 0-100: {len(bad)}")
print(bad[['sid', 'end_date', 'other_pct']].head(10).to_string(index=False))

# Check all percentage columns for out-of-range values
print("\n=== All pct columns — out of 0-100 range ===")
pct_cols = ['promoter_pct', 'fii_pct', 'mf_pct', 'dii_pct', 'public_pct', 
            'pledge_pct', 'insurance_pct', 'retail_hni_pct', 'other_pct']
for col in pct_cols:
    bad_count = (~sh_clean[col].between(0, 100, inclusive='both')).sum()
    if bad_count > 0:
        rng = sh_clean[col].describe()[['min', 'max']]
        print(f"  {col:20s}  {bad_count} bad rows  min={rng['min']:.2f} max={rng['max']:.2f}")


In [ ]:
# Fix: clip floating-point noise to 0
pct_cols = ['promoter_pct', 'fii_pct', 'mf_pct', 'dii_pct', 'public_pct', 
            'pledge_pct', 'insurance_pct', 'retail_hni_pct', 'other_pct']
for col in pct_cols:
    sh_insert[col] = sh_insert[col].clip(lower=0, upper=100)

sh_rows = upsert_df(sh_insert, "shareholding")
print(f"shareholding: {sh_rows} rows inserted")

# Quick verification of all 4 tables
from db import table_counts
table_counts()
